In [ ]:
# ============================================================================
# PRODUCTION IR DOCUMENT EXTRACTOR - FULLY OPTIMIZED
# ============================================================================

import re
import random
import time
from datetime import datetime
from typing import List, Dict, Any, Optional, Set, Tuple
from dataclasses import dataclass, asdict
from urllib.parse import urljoin, urlparse, unquote, parse_qs, urlunparse
import json
import os
import hashlib
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import logging

# Auto-install ChromeDriver
from webdriver_manager.chrome import ChromeDriverManager

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class DocumentInfo:
    """Structure to hold document information"""
    title: str
    url: str
    normalized_url: str  # URL without parameters for deduplication
    document_type: str
    publication_date: Optional[str]
    file_extension: str
    relevance_score: float
    content_preview: str
    metadata: Dict[str, Any]
    file_size: Optional[int] = None
    is_verified: bool = False
    download_url: Optional[str] = None

class ProductionIRExtractor:
    """
    Production-ready IR document extractor with all optimizations
    """
    
    def __init__(self, debug: bool = True):
        self.debug = debug
        self.file_extensions = ['.pdf', '.xlsx', '.xls', '.pptx', '.ppt', '.docx', '.doc', '.csv']
        self.seen_urls: Set[str] = set()  # Stores normalized URLs
        self.driver = None
        
        # Financial vocabulary for semantic scoring
        self.financial_vocabulary = {
            'earnings': 10.0, 'quarterly': 9.0, 'results': 8.0, 'financial': 8.0,
            'report': 7.0, 'filing': 8.0, 'presentation': 7.0, 'transcript': 7.0,
            'q1': 6.0, 'q2': 6.0, 'q3': 6.0, 'q4': 6.0, 'quarter': 6.0,
            'annual': 7.0, '10-k': 9.0, '10-q': 9.0, '8-k': 8.0, 'sec': 7.0,
            'slide': 5.0, 'deck': 5.0, 'webcast': 6.0, 'call': 6.0,
            'supplemental': 7.0, 'investor': 8.0, '2024': 8.0, '2025': 10.0,
        }
        
        # Navigation targets (will explore multiple)
        self.navigation_targets = [
            'quarterly results', 'quarterly earnings', 'financial results',
            'sec filings', 'reports', 'press releases', 'presentations',
            'investor relations', 'financial information'
        ]
    
    def _setup_selenium(self):
        """Setup Selenium with auto-installed ChromeDriver"""
        if self.debug:
            print("   🔧 Auto-installing ChromeDriver...")
        
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        
        # Auto-install ChromeDriver using webdriver-manager
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        
        driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
            'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
        })
        
        if self.debug:
            print("   ✅ ChromeDriver ready")
        
        return driver
    
    def _normalize_url(self, url: str) -> str:
        """
        Normalize URL by removing tracking parameters
        Example: file.pdf?version=2&utm_source=email -> file.pdf
        """
        parsed = urlparse(url)
        
        # Remove query parameters that are just tracking/versioning
        query_params = parse_qs(parsed.query)
        
        # Keep only meaningful parameters
        important_params = ['id', 'document', 'file']
        filtered_params = {k: v for k, v in query_params.items() if k.lower() in important_params}
        
        # Reconstruct URL without tracking params
        normalized = urlunparse((
            parsed.scheme,
            parsed.netloc,
            parsed.path,
            parsed.params,
            '',  # No query string or only important params
            ''   # No fragment
        ))
        
        return normalized
    
    async def extract_all_documents_from_ir_page(
        self, 
        ir_url: str, 
        ticker: str, 
        company_name: str
    ) -> List[DocumentInfo]:
        """Main extraction with smart navigation"""
        if self.debug:
            print(f"\n🔍 [{ticker}] {company_name}")
            print(f"   📍 URL: {ir_url}")
        
        documents = []
        visited_pages = set()
        
        try:
            if not self.driver:
                self.driver = self._setup_selenium()
            
            # Extract from main page
            self.driver.get(ir_url)
            time.sleep(random.uniform(3, 5))
            
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            
            soup = BeautifulSoup(self.driver.page_source, 'html.parser')
            main_docs = self._extract_with_semantic_ai(soup, ir_url, ticker, company_name)
            documents.extend(main_docs)
            visited_pages.add(ir_url)
            
            if self.debug:
                print(f"   📄 Main page: {len(main_docs)} documents")
            
            # Smart navigation: Explore ALL promising subpages (no limit)
            navigation_links = self._find_promising_subpages(soup, ir_url)
            
            explored_count = 0
            max_subpages = len(navigation_links)  # Explore ALL, no limit
            
            for nav_url, nav_score in navigation_links:
                if explored_count >= max_subpages:
                    break
                
                if nav_url in visited_pages:
                    continue
                
                try:
                    if self.debug:
                        print(f"   🔗 [{explored_count + 1}/{len(navigation_links)}] {nav_url[:70]}...")
                    
                    self.driver.get(nav_url)
                    time.sleep(random.uniform(2, 4))
                    
                    sub_soup = BeautifulSoup(self.driver.page_source, 'html.parser')
                    sub_docs = self._extract_with_semantic_ai(sub_soup, nav_url, ticker, company_name)
                    
                    documents.extend(sub_docs)
                    visited_pages.add(nav_url)
                    explored_count += 1
                    
                    if self.debug:
                        if sub_docs:
                            print(f"   ✅ Found {len(sub_docs)} documents")
                        else:
                            print(f"   ⚠️  No documents found")
                
                except Exception as e:
                    if self.debug:
                        print(f"   ❌ Failed: {str(e)[:50]}")
                    continue
            
        except Exception as e:
            if self.debug:
                print(f"   ❌ Error: {str(e)[:80]}")
        
        # Clean, deduplicate, and rank
        documents = self._remove_duplicates_smart(documents)
        documents = self._rank_intelligently(documents)
        
        # Filter to get only latest relevant documents
        documents = self._filter_to_latest_documents(documents)
        
        if self.debug:
            print(f"   ✅ Final: {len(documents)} latest documents")
            for i, doc in enumerate(documents[:5], 1):
                print(f"      {i}. [{doc.document_type}] {doc.title[:50]}...")
        
        return documents
    
    def _find_promising_subpages(self, soup: BeautifulSoup, base_url: str) -> List[Tuple[str, float]]:
        """
        Find 3-5 most promising subpages to explore
        Returns: List of (url, score) tuples sorted by promise
        """
        subpages = []
        
        # Find all navigation and content links
        all_links = soup.find_all('a', href=True)
        
        for link in all_links:
            try:
                href = link.get('href', '')
                if not href or href.startswith('#') or href.startswith('javascript:'):
                    continue
                
                full_url = urljoin(base_url, href)
                
                # Must be same domain
                if urlparse(full_url).netloc != urlparse(base_url).netloc:
                    continue
                
                # Get link text and context
                text = link.get_text(strip=True).lower()
                title = link.get('title', '').lower()
                
                # Calculate promise score
                promise_score = 0.0
                
                # High-value navigation targets
                for target in self.navigation_targets:
                    if target in text or target in title:
                        promise_score += 10.0
                
                # Boost for specific keywords
                if any(kw in text for kw in ['earnings', 'results', 'quarterly']):
                    promise_score += 8.0
                if any(kw in text for kw in ['sec', 'filing', 'reports']):
                    promise_score += 7.0
                if any(kw in text for kw in ['presentation', 'investor']):
                    promise_score += 6.0
                
                if promise_score >= 6.0:
                    subpages.append((full_url, promise_score))
            
            except:
                continue
        
        # Remove duplicates and sort by score
        unique_subpages = {}
        for url, score in subpages:
            if url not in unique_subpages or score > unique_subpages[url]:
                unique_subpages[url] = score
        
        sorted_subpages = sorted(unique_subpages.items(), key=lambda x: x[1], reverse=True)
        
        if self.debug and sorted_subpages:
            print(f"   🗺️  Found {len(sorted_subpages)} promising subpages (will explore ALL)")
        
        return sorted_subpages  # Return ALL, no [:5] limit
    
    def _extract_with_semantic_ai(
        self, 
        soup: BeautifulSoup, 
        base_url: str, 
        ticker: str, 
        company_name: str
    ) -> List[DocumentInfo]:
        """Extract using semantic similarity"""
        documents = []
        all_links = soup.find_all('a', href=True)
        
        for link in all_links:
            try:
                href = link.get('href', '')
                if not href or href.startswith('#'):
                    continue
                
                full_url = urljoin(base_url, href)
                normalized_url = self._normalize_url(full_url)
                
                # Check if already seen (using normalized URL)
                if normalized_url in self.seen_urls:
                    continue
                
                # Get text and context
                text = link.get_text(strip=True)
                title = link.get('title', '')
                aria = link.get('aria-label', '')
                parent_text = self._get_parent_context(link)
                
                combined_text = f"{text} {title} {aria} {parent_text}"
                
                # Calculate semantic score
                semantic_score = self._calculate_semantic_score(combined_text, full_url)
                
                # LOWER threshold to capture more documents (was 3.0, now 2.0)
                if semantic_score > 2.0:  # More lenient threshold
                    self.seen_urls.add(normalized_url)
                    
                    doc_title = self._extract_smart_title(link, text, title, full_url)
                    doc_type = self._classify_semantically(doc_title, full_url, combined_text)
                    pub_date = self._extract_date_intelligently(combined_text + full_url)
                    
                    doc = DocumentInfo(
                        title=doc_title[:200],
                        url=full_url,
                        normalized_url=normalized_url,
                        document_type=doc_type,
                        publication_date=pub_date,
                        file_extension=self._get_extension(full_url),
                        relevance_score=semantic_score * 10,
                        content_preview=doc_title[:150],
                        metadata={
                            'ticker': ticker,
                            'company': company_name,
                            'extraction_method': 'semantic_ai'
                        },
                        is_verified=True,
                        download_url=full_url
                    )
                    
                    documents.append(doc)
            
            except:
                continue
        
        return documents
    
    def _calculate_semantic_score(self, text: str, url: str) -> float:
        """Calculate semantic similarity score"""
        combined = (text + ' ' + url).lower()
        tokens = re.findall(r'\w+', combined)
        
        score = 0.0
        for token in tokens:
            if token in self.financial_vocabulary:
                score += self.financial_vocabulary[token]
        
        if len(tokens) > 0:
            score = score / len(tokens) * 10
        
        # Boost for file extensions
        if any(ext in url.lower() for ext in self.file_extensions):
            score *= 1.5
        
        # Boost for multiple financial terms
        unique_terms = sum(1 for token in set(tokens) if token in self.financial_vocabulary)
        if unique_terms >= 3:
            score *= 1.3
        
        # LOWER threshold for "latest" - accept more documents
        return score
    
    def _get_parent_context(self, element, max_depth: int = 3) -> str:
        """Get context from parent elements"""
        context = []
        current = element.parent
        
        for _ in range(max_depth):
            if not current:
                break
            
            heading = current.find(['h1', 'h2', 'h3', 'h4'])
            if heading:
                context.append(heading.get_text(strip=True))
            
            time_elem = current.find('time')
            if time_elem:
                context.append(time_elem.get_text(strip=True))
            
            current = current.parent
        
        return ' '.join(context)
    
    def _extract_smart_title(self, link, text: str, title_attr: str, url: str) -> str:
        """Extract best title from multiple sources"""
        candidates = []
        
        if text and len(text) > 10:
            cleaned = self._clean_text(text)
            if cleaned and len(cleaned) > 10:
                candidates.append((cleaned, 3.0))
        
        if title_attr and len(title_attr) > 10:
            cleaned = self._clean_text(title_attr)
            if cleaned:
                candidates.append((cleaned, 2.5))
        
        try:
            parent = link.parent
            for _ in range(3):
                if not parent:
                    break
                heading = parent.find(['h1', 'h2', 'h3', 'h4'])
                if heading:
                    h_text = heading.get_text(strip=True)
                    if h_text and len(h_text) > 10:
                        candidates.append((h_text, 2.0))
                        break
                parent = parent.parent
        except:
            pass
        
        filename = self._extract_filename_smart(url)
        if filename and len(filename) > 5:
            candidates.append((filename, 1.5))
        
        if candidates:
            candidates.sort(key=lambda x: x[1], reverse=True)
            return candidates[0][0]
        
        return 'Financial Document'
    
    def _extract_filename_smart(self, url: str) -> str:
        """Extract clean filename from URL"""
        parsed = urlparse(url)
        filename = os.path.basename(unquote(parsed.path))
        
        filename = re.sub(r'[-_]', ' ', filename)
        filename = re.sub(r'\.(pdf|xlsx?|pptx?|docx?)$', '', filename, flags=re.I)
        filename = re.sub(r'\s+', ' ', filename).strip()
        filename = re.sub(r'\b[a-f0-9]{8,}\b', '', filename, flags=re.I)
        filename = re.sub(r'\s+', ' ', filename).strip()
        
        return filename
    
    def _clean_text(self, text: str) -> str:
        """Clean text"""
        if not text:
            return ''
        
        text = re.sub(r'\(opens?\s+in.*?\)', '', text, flags=re.I)
        text = re.sub(r'\(pdf\)', '', text, flags=re.I)
        text = re.sub(r'\s+', ' ', text).strip()
        
        generic = ['click here', 'download', 'read more', 'view', 'details', 'default']
        if text.lower().strip() in generic:
            return ''
        
        return text
    
    def _classify_semantically(self, title: str, url: str, context: str) -> str:
        """Classify document type"""
        combined = f"{title} {url} {context}".lower()
        
        categories = {
            'earnings_release': ['earnings', 'results', 'announces', 'reports', 'quarterly', 'press', 'release'],
            'presentation': ['presentation', 'slides', 'deck', 'investor', 'day'],
            'transcript': ['transcript', 'call', 'remarks', 'conference'],
            'annual_report': ['annual', '10-k', 'yearly'],
            'quarterly_report': ['10-q', 'quarterly'],
            'sec_filing': ['8-k', 'filing', 'form', 'sec'],
            'supplemental_data': ['supplemental', 'data', 'tables', 'statistics'],
        }
        
        best_category = 'financial_document'
        best_score = 0
        
        for category, terms in categories.items():
            score = sum(1 for term in terms if term in combined)
            if score > best_score:
                best_score = score
                best_category = category
        
        if '.pdf' in url and best_category == 'financial_document':
            return 'pdf_document'
        elif '.xls' in url:
            return 'excel_data'
        elif '.ppt' in url:
            return 'presentation'
        
        return best_category
    
    def _extract_date_intelligently(self, text: str) -> Optional[str]:
        """Extract date"""
        patterns = [
            r'Q([1-4])\s*20([12]\d)',
            r'20([12]\d)\s*Q([1-4])',
            r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{1,2},?\s+20[12]\d',
            r'\d{1,2}[/-]\d{1,2}[/-]20[12]\d',
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.I)
            if match:
                return match.group(0)
        
        return None
    
    def _get_extension(self, url: str) -> str:
        """Get file extension"""
        url_lower = url.lower()
        for ext in self.file_extensions:
            if ext in url_lower:
                return ext
        return '.unknown'
    
    def _remove_duplicates_smart(self, documents: List[DocumentInfo]) -> List[DocumentInfo]:
        """
        Smart duplicate removal using normalized URLs
        """
        seen_normalized = set()
        unique = []
        
        for doc in documents:
            if doc.normalized_url not in seen_normalized:
                seen_normalized.add(doc.normalized_url)
                unique.append(doc)
        
        if self.debug and len(documents) > len(unique):
            removed = len(documents) - len(unique)
            print(f"   🧹 Removed {removed} duplicates")
        
        return unique
    
    def _filter_to_latest_documents(self, documents: List[DocumentInfo]) -> List[DocumentInfo]:
        """
        Filter to keep only:
        1. Latest document of each type (earnings_release, presentation, etc.)
        2. All quarters from the most recent year (Q1, Q2, Q3, Q4 of latest year)
        """
        if not documents:
            return documents
        
        # Step 1: Extract years and quarters from documents
        docs_with_parsed_dates = []
        for doc in documents:
            year, quarter = self._parse_year_quarter(doc.publication_date, doc.title, doc.url)
            docs_with_parsed_dates.append({
                'doc': doc,
                'year': year,
                'quarter': quarter
            })
        
        # Step 2: Find the most recent year
        years = [d['year'] for d in docs_with_parsed_dates if d['year']]
        most_recent_year = max(years) if years else datetime.now().year
        
        if self.debug:
            print(f"   📅 Most recent year found: {most_recent_year}")
        
        # Step 3: Group documents by type
        by_type = {}
        for item in docs_with_parsed_dates:
            doc = item['doc']
            doc_type = doc.document_type
            
            if doc_type not in by_type:
                by_type[doc_type] = []
            by_type[doc_type].append(item)
        
        # Step 4: Filter strategy
        filtered_docs = []
        
        for doc_type, items in by_type.items():
            # Sort by year, then quarter (descending)
            items_sorted = sorted(
                items,
                key=lambda x: (
                    x['year'] or 0,
                    x['quarter'] or 0,
                    x['doc'].relevance_score
                ),
                reverse=True
            )
            
            # Strategy depends on document type
            if doc_type in ['earnings_release', 'presentation', 'transcript']:
                # For quarterly documents: Keep all quarters from most recent year
                latest_year_docs = [
                    item['doc'] for item in items_sorted 
                    if item['year'] == most_recent_year
                ]
                
                if latest_year_docs:
                    # Group by quarter and keep one per quarter
                    by_quarter = {}
                    for item in items_sorted:
                        if item['year'] == most_recent_year and item['quarter']:
                            q = item['quarter']
                            if q not in by_quarter:
                                by_quarter[q] = item['doc']
                    
                    filtered_docs.extend(by_quarter.values())
                    
                    if self.debug:
                        quarters = sorted(by_quarter.keys())
                        print(f"   📊 {doc_type}: Keeping {len(by_quarter)} quarters from {most_recent_year} {quarters}")
                else:
                    # Fallback: keep latest document
                    if items_sorted:
                        filtered_docs.append(items_sorted[0]['doc'])
                        if self.debug:
                            print(f"   📄 {doc_type}: Keeping 1 latest document")
            
            elif doc_type in ['annual_report', 'sec_filing', 'quarterly_report']:
                # For annual/filing documents: Keep only the absolute latest
                if items_sorted:
                    filtered_docs.append(items_sorted[0]['doc'])
                    year = items_sorted[0]['year']
                    if self.debug:
                        print(f"   📄 {doc_type}: Keeping latest ({year})")
            
            elif doc_type in ['supplemental_data', 'excel_data']:
                # For supplemental data: Keep latest from most recent year
                latest_year_items = [
                    item for item in items_sorted 
                    if item['year'] == most_recent_year or not item['year']
                ]
                if latest_year_items:
                    filtered_docs.append(latest_year_items[0]['doc'])
                    if self.debug:
                        print(f"   📄 {doc_type}: Keeping 1 latest from {most_recent_year}")
            
            else:
                # For other types: Keep only the latest
                if items_sorted:
                    filtered_docs.append(items_sorted[0]['doc'])
                    if self.debug:
                        print(f"   📄 {doc_type}: Keeping 1 latest")
        
        return filtered_docs
    
    def _parse_year_quarter(self, pub_date: Optional[str], title: str, url: str) -> Tuple[Optional[int], Optional[int]]:
        """
        Parse year and quarter from publication date, title, or URL
        Returns: (year, quarter) where quarter is 1-4 or None
        """
        combined = f"{pub_date or ''} {title} {url}".lower()
        
        year = None
        quarter = None
        
        # Extract year (2020-2026)
        year_match = re.search(r'20(2[0-6]|1[0-9])', combined)
        if year_match:
            year = int(year_match.group(0))
        
        # Extract quarter
        quarter_patterns = [
            (r'q1\b', 1), (r'\bq1\b', 1), (r'first.{0,10}quarter', 1),
            (r'q2\b', 2), (r'\bq2\b', 2), (r'second.{0,10}quarter', 2),
            (r'q3\b', 3), (r'\bq3\b', 3), (r'third.{0,10}quarter', 3),
            (r'q4\b', 4), (r'\bq4\b', 4), (r'fourth.{0,10}quarter', 4),
        ]
        
        for pattern, q_num in quarter_patterns:
            if re.search(pattern, combined, re.I):
                quarter = q_num
                break
        
        # If we found Q but no year, try to infer from context
        if quarter and not year:
            # Look for year near the quarter mention
            context_match = re.search(rf'q{quarter}.{{0,20}}(20\d{{2}})', combined, re.I)
            if context_match:
                year = int(context_match.group(1))
        
        return year, quarter
    
    def _rank_intelligently(self, documents: List[DocumentInfo]) -> List[DocumentInfo]:
        """Rank documents by relevance"""
        def rank_score(doc):
            score = doc.relevance_score
            
            current_year = datetime.now().year
            if doc.publication_date:
                if str(current_year) in doc.publication_date:
                    score += 30
                elif str(current_year - 1) in doc.publication_date:
                    score += 20
            
            if doc.file_extension in ['.pdf', '.xlsx', '.pptx']:
                score += 15
            
            if doc.document_type in ['earnings_release', 'presentation', 'sec_filing']:
                score += 20
            
            return score
        
        return sorted(documents, key=rank_score, reverse=True)
    
    def close(self):
        """Close driver"""
        if self.driver:
            self.driver.quit()
            self.driver = None


# ============================================================================
# Main Execution Functions
# ============================================================================

async def extract_documents_production(
    companies: List[Dict[str, str]],
    debug: bool = True
) -> Dict[str, List[DocumentInfo]]:
    """Extract documents from all companies"""
    extractor = ProductionIRExtractor(debug=debug)
    all_documents = {}
    
    print("\n" + "="*80)
    print("PRODUCTION IR DOCUMENT EXTRACTION - LATEST DOCUMENTS ONLY")
    print("✅ Auto ChromeDriver • Latest of each type • All quarters from recent year")
    print("✅ Smart filtering: Most relevant & recent documents")
    print("="*80)
    
    try:
        for i, company in enumerate(companies, 1):
            ticker = company['ticker']
            company_name = company['company_name']
            ir_url = company.get('investor_relations_url', '')
            
            print(f"\n[{i}/{len(companies)}] {ticker}")
            
            try:
                documents = await extractor.extract_all_documents_from_ir_page(
                    ir_url, ticker, company_name
                )
                all_documents[ticker] = documents
                time.sleep(random.uniform(2, 4))
            except Exception as e:
                if debug:
                    print(f"   ❌ Error: {str(e)[:60]}")
                all_documents[ticker] = []
    finally:
        extractor.close()
    
    print("\n" + "="*80)
    print("EXTRACTION SUMMARY")
    print("="*80)
    
    total = sum(len(docs) for docs in all_documents.values())
    with_docs = sum(1 for docs in all_documents.values() if docs)
    avg = total / len(companies) if companies else 0
    
    print(f"✅ Total documents: {total}")
    print(f"✅ Companies with documents: {with_docs}/{len(companies)}")
    print(f"✅ Average per company: {avg:.1f}")
    
    return all_documents


def export_to_json(all_documents: Dict[str, List[DocumentInfo]], output_path: str):
    """Export to JSON"""
    export_data = {}
    for ticker, documents in all_documents.items():
        export_data[ticker] = [asdict(doc) for doc in documents]
    
    with open(output_path, 'w') as f:
        json.dump(export_data, f, indent=2, default=str)
    
    print(f"\n✅ Exported to {output_path}")


async def main(json_path: str = None):
    """Main entry point"""
    possible_paths = [
        json_path,
        "../data/catalogue/cnbc_companies_with_ir.json",
        "/Users/RiyanshiKedia/Documents/GitHub/investment-report-extractor/data/catalogue/cnbc_companies_with_ir.json",
    ]
    
    companies_file = None
    for path in possible_paths:
        if path and os.path.exists(path):
            companies_file = path
            break
    
    if not companies_file:
        raise FileNotFoundError("Could not find cnbc_companies_with_ir.json")
    
    with open(companies_file, 'r') as f:
        companies = json.load(f)
    
    all_documents = await extract_documents_production(companies, debug=True)
    
    output_dir = os.path.join(os.path.dirname(companies_file), "../documents")
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = os.path.join(output_dir, "production_documents_optimized.json")
    export_to_json(all_documents, output_file)
    
    return all_documents


async def test_extraction(num_companies: int = 5):
    """Quick test"""
    json_path = "/Users/RiyanshiKedia/Documents/GitHub/investment-report-extractor/data/catalogue/cnbc_companies_with_ir.json"
    
    with open(json_path, 'r') as f:
        companies = json.load(f)
    
    test_companies = companies[:num_companies]
    
    print(f"🧪 Testing OPTIMIZED extraction on {len(test_companies)} companies")
    
    all_documents = await extract_documents_production(test_companies, debug=True)
    
    print("\n" + "="*80)
    print("TEST RESULTS")
    print("="*80)
    
    for ticker, docs in all_documents.items():
        print(f"\n{ticker}: {len(docs)} documents")
        for i, doc in enumerate(docs[:5], 1):
            print(f"  {i}. [{doc.document_type}] {doc.title[:60]}")
    
    return all_documents


# ============================================================================

# TO RUN:
# test_results = await test_extraction(num_companies=5)
# all_results = await main()
# ============================================================================

In [ ]:
#test_results = await test_extraction(num_companies=5)

2025-10-08 17:03:37,867 - INFO - ====== WebDriver manager ======


🧪 Testing OPTIMIZED extraction on 5 companies

PRODUCTION IR DOCUMENT EXTRACTION - LATEST DOCUMENTS ONLY
✅ Auto ChromeDriver • Latest of each type • All quarters from recent year
✅ Smart filtering: Most relevant & recent documents

[1/5] AMGN

🔍 [AMGN] AMGN
   📍 URL: http://investors.amgen.com/
   🔧 Auto-installing ChromeDriver...


2025-10-08 17:03:38,134 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-08 17:03:38,369 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-08 17:03:38,533 - INFO - Driver [/Users/RiyanshiKedia/.wdm/drivers/chromedriver/mac64/141.0.7390.65/chromedriver-mac-arm64/chromedriver] found in cache


   ✅ ChromeDriver ready
   📄 Main page: 17 documents
   🗺️  Found 12 promising subpages (will explore ALL)
   🔗 [1/12] http://investors.amgen.com/news-releases/news-release-details/amgen-re...
   ✅ Found 2 documents
   🔗 [2/12] http://investors.amgen.com/financials/quarterly-earnings...
   ✅ Found 112 documents
   🔗 [3/12] http://investors.amgen.com/financials/sec-filings...
   ✅ Found 11 documents
   🔗 [4/12] http://investors.amgen.com/financials/annual-reports...
   ✅ Found 25 documents
   🔗 [5/12] http://investors.amgen.com/news-and-events/presentations...
   ✅ Found 6 documents
   🔗 [6/12] http://investors.amgen.com/static-files/27fcb898-9cee-48db-9684-1da258...
   ⚠️  No documents found
   🔗 [7/12] http://investors.amgen.com/news-and-events/press-releases...
   ✅ Found 17 documents
   🔗 [8/12] http://investors.amgen.com/forward-looking-statement?evturl=https://in...
   ✅ Found 1 documents
   🔗 [9/12] http://investors.amgen.com/news-releases/news-release-details/amgen-an...
   ✅ Fo